# [4.1] Subtitle Batch 2 (Tải Trực Tiếp Từ Kaggle Dataset) → Volume `keyframes-batch-2-v2`

`docs/triet/ke-hoach-offline-ingest.md` §4.4, §4.5 · config `config/ingest_shards_batch2_btc_subtitles.yaml`

Chạy sau `02-keyframes-btc-import.ipynb`. **Cùng cách chia người** với 02 và 03.

| Tầng | Model | Ra artifact |
|---|---|---|
| `n2_subtitles` | `faster-whisper large-v3` | `subtitle_segments` (thô) + `subtitles` (theo keyframe) |

### Khác biệt so với Notebook 04 cũ (Batch 1):
1. **Không dùng CLAP (n3_audio)**: Loại bỏ hoàn toàn mô hình CLAP để tập trung 100% sức mạnh GPU cho Whisper và tiết kiệm thời gian.
2. **Đọc trực tiếp từ Kaggle Dataset**: Các video `.mp4` và `.mov` của Batch 2 đã được tải lên thành Kaggle Dataset (vào thư mục `/kaggle/input/`). Code sẽ tự động lùng sục file video local thay vì dùng wget tải từ mạng.
3. **Chống lỗi Video Câm**: Camera giao thông (Video N, đuôi `.mov`) thường không có âm thanh. Nếu `ffmpeg` không tìm thấy luồng âm thanh, code sẽ tự động trả về mảng rỗng (0 dòng phụ đề) thay vì ném lỗi đánh sập tiến trình.

> **⚠️ CHÚ Ý QUAN TRỌNG:**
> Bạn BẮT BUỘC phải nhấn **Add Input** trên Kaggle và chèn TẤT CẢ các Dataset chứa Video Batch 2 vào Notebook này trước khi chạy. Code sẽ quét qua toàn bộ `/kaggle/input/` để tìm video được phân công.


## Step 0a — repo + cài đặt

In [ ]:
MEMBER      = "trong"          # <-- ĐỔI THÀNH TÊN BẠN
SHARDS      = "config/ingest_shards_batch2_btc_subtitles.yaml"
TRANSPORT   = "modal"
VOLUME      = "keyframes-batch-2-v2"  # CÙNG volume với notebook 02 và 03
VOL_VERSION = 2
KEEP_MODELS = True
REPO_URL    = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF    = "feature/triet/structure"

import os, subprocess, sys
from pathlib import Path

REPO = Path("/kaggle/input/mfe-repo")
if not REPO.is_dir():
    REPO = Path("/kaggle/working/mfe")
    if not REPO.is_dir():
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF, url, str(REPO)],
                       check=True, capture_output=True)
sys.path.insert(0, str(REPO / "src"))

req = REPO / "notebooks" / "requirements-kaggle.lock.txt"
if not req.is_file():
    req = REPO / "notebooks" / "requirements-kaggle-btc-video.txt"
print(f"cài từ {req.name}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)


## Step 0b — Vá Lỗi (Monkey-Patch), Khởi tạo Modal, Lên Lịch Việc

In [ ]:
import logging, yaml, csv, subprocess
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from kaggle_secrets import UserSecretsClient
from mfe.offline.bundle import BundleLayout, BundleWriter, StageStatus, load_shard_plan
from mfe.offline.stages import STAGES, StageContext, Keyframe, unload_models
from mfe.offline.transport import build_transport
import modal

# 1. VÁ LỖI LÀM TRÒN SỐ (FLOAT) KHI ĐỌC KEYFRAME MAP
def read_keyframes_patched(map_csv, keyframes_dir):
    rows = list(csv.DictReader(map_csv.open(encoding="utf-8")))
    out = []
    for row in rows:
        n = int(row["n"])
        pts_time, fps = float(row["pts_time"]), float(row["fps"])
        frame_idx = int(row["frame_idx"])
        out.append(Keyframe(n, pts_time, fps, frame_idx, keyframes_dir / f"{n:04d}.jpg"))
    return out

# 2. VÁ LỖI VIDEO CÂM (KHÔNG CÓ AUDIO TRACK) CỦA STAGECONTEXT
# Bỏ qua lỗi ffmpeg nếu file video (VD: .mov của camera giao thông) không chứa âm thanh
def patched_audio_path(self) -> Path:
    wav = self.workdir / f"{self.video_id}.wav"
    if not wav.is_file():
        # Thử trích xuất âm thanh
        result = subprocess.run(
            ["ffmpeg", "-nostdin", "-y", "-i", str(self.video_path),
             "-vn", "-ac", "1", "-ar", "48000", str(wav)],
            capture_output=True,
        )
        if result.returncode != 0:
            # Nếu ffmpeg vấp ngã (không thấy luồng audio), tạo 1 file wav câm (dummy) hoặc trả về None
            # Tuy nhiên, Whisper không thích chạy file câm (có thể gây lỗi thư viện).
            # Ta sẽ tạo cờ báo hiệu video này bị CÂM.
            setattr(self, "_is_silent_video", True)
            return None
    setattr(self, "_is_silent_video", False)
    return wav

StageContext.audio_path = patched_audio_path

# 3. VÁ TẦNG WHISPER (BỎ QUA NẾU VIDEO CÂM)
original_stage_n2 = STAGES["n2_subtitles"]
def stage_n2_patched(ctx):
    # Gọi thử audio_path để kích hoạt cơ chế extract
    wav = ctx.audio_path()
    if getattr(ctx, "_is_silent_video", False) or wav is None:
        print(f"      [CẢNH BÁO] {ctx.video_id} KHÔNG có luồng âm thanh. Bỏ qua Whisper.")
        # Lưu segments rỗng
        segments_path = ctx.workdir / f"segments_{ctx.video_id}.json"
        import json
        segments_path.write_text(json.dumps({"language": "vi", "segments": []}, ensure_ascii=False, indent=1), encoding="utf-8")
        ctx.writer.write_file(segments_path, ctx.layout.subtitle_segments(ctx.video_id))
        
        # Ghi artifact subtitles rỗng
        from mfe.offline.stages import Stopwatch
        with Stopwatch() as timer:
            pass
        return ctx.writer.write_text_part(
            "subtitles", ctx.video_id, {},
            model="faster-whisper large-v3",
            params={"beam_size": 5, "language": "vi", "window": "next_keyframe", "max_empty_ratio": 0.999},
            facts={"segments": 0, "keyframes_total": len(ctx.keyframes), "keyframes_with_subtitle": 0, "keyframes_without_subtitle": len(ctx.keyframes), "empty_ratio": 1.0},
            elapsed_s=timer.seconds,
        )
    
    # Nếu có tiếng, chạy Whisper gốc
    return original_stage_n2(ctx)

STAGES["n2_subtitles"] = stage_n2_patched

# --- KHỞI TẠO MÔI TRƯỜNG & LÊN LỊCH ---
secrets = UserSecretsClient()
os.environ["MODAL_TOKEN_ID"] = secrets.get_secret("MODAL_TOKEN_ID")
os.environ["MODAL_TOKEN_SECRET"] = secrets.get_secret("MODAL_TOKEN_SECRET")

RAW       = yaml.safe_load((REPO / SHARDS).read_text(encoding="utf-8"))
plan      = load_shard_plan(REPO / SHARDS)
layout    = BundleLayout(plan.bundle)
transport = build_transport(TRANSPORT, volume=VOLUME,
                            local_root="/kaggle/working/bundle_local",
                            version=VOL_VERSION)
WORK   = Path("/kaggle/working/scratch"); WORK.mkdir(parents=True, exist_ok=True)
writer = BundleWriter(transport, layout, MEMBER, WORK)

assert plan.keyframe_set == "btc"
STAGE_LIST = ["n2_subtitles"] # CHỈ CHẠY MỖI N2
PARAMS     = RAW.get("params", {})

MY_VIDEOS = plan.videos_for(MEMBER)
status    = StageStatus.load(transport, layout, MEMBER)

todo = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not status.is_done(s, v)]

print(f"bundle  : {plan.bundle}  ->  volume {VOLUME} (v{VOL_VERSION})")
print(f"người   : {MEMBER}")
print(f"video   : {len(MY_VIDEOS)} (tổng {len(plan.videos)})")
print(f"tầng    : {', '.join(STAGE_LIST)}")
print(f"còn lại : {len(todo)} việc / {len(MY_VIDEOS) * len(STAGE_LIST)}")


In [ ]:
import glob

def find_local_video(video_id):
    """Lùng sục mọi ngóc ngách trong /kaggle/input để tìm file video .mp4 hoặc .mov"""
    # Tìm chính xác theo tên file chứa video_id
    search_patterns = [
        f"/kaggle/input/**/*{video_id}.mp4",
        f"/kaggle/input/**/*{video_id}.mov",
        f"/kaggle/input/**/*{video_id}*.mp4",
        f"/kaggle/input/**/*{video_id}*.mov"
    ]
    
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            # Loại trừ các file rác sinh ra bởi hệ điều hành (như ._S01-V001.mp4)
            valid_matches = [m for m in matches if "/._" not in m]
            if valid_matches:
                return Path(valid_matches[0])
                
    raise FileNotFoundError(f"KHÔNG TÌM THẤY VIDEO: {video_id} trong bất kỳ Dataset nào. Đã Add Input chưa?")


## Step 1 — Vòng lặp chính

Mỗi video:
1. Lấy Map CSV từ Modal Volume
2. Tìm File `.mp4` hoặc `.mov` trong Dataset
3. Chạy Faster-Whisper trích xuất phụ đề
4. Đẩy kết quả Json lên Modal


In [ ]:
import time, traceback
from tqdm.auto import tqdm

failures = []
started = time.perf_counter()
done_count = 0

pending = [v for v in MY_VIDEOS if not status.is_done("n2_subtitles", v)]

print("Kết nối Modal Volume để kéo Map CSV...")
vol = modal.Volume.from_name(VOLUME)

if not pending:
    print(f"=== Không còn việc cần chạy cho {MEMBER} ===")
else:
    print(f"\n=== Còn {len(pending)}/{len(MY_VIDEOS)} video cần chạy Subtitle ===")

for video_id in tqdm(pending, desc="Tổng Tiến Trình Video"):
    wav_path = WORK / f"{video_id}.wav"
    map_csv = WORK / "map-keyframes" / f"{video_id}.csv"
    
    try:
        # 1. Tìm Video Local
        video_path = find_local_video(video_id)
        file_size_mb = video_path.stat().st_size / 2**20
        print(f"\n  Đang chạy {video_id} ({file_size_mb:.1f} MB) từ: {video_path.parent.name}")
        
        # 2. Lấy Map CSV từ Modal
        remote_csv = f"bundle/{plan.bundle}/map-keyframes/{video_id}.csv"
        map_csv.parent.mkdir(parents=True, exist_ok=True)
        with map_csv.open("wb") as f:
            for chunk in vol.read_file(remote_csv):
                f.write(chunk)
        
        keyframes = read_keyframes_patched(map_csv, Path("/dev/null"))
        
        # 3. Chạy N2_Subtitles
        ctx = StageContext(video_id, video_path, keyframes, writer, layout, WORK)
        ctx.params = dict(PARAMS.get("n2_subtitles", {}))
        
        marker = STAGES["n2_subtitles"](ctx)
        status.done.add(("n2_subtitles", video_id))
        print(f"      n2_subtitles     {marker.get('rows', '-'):>6} dòng  {marker.get('elapsed_s')}s")

        # 4. Đóng gói & Xóa rác
        writer.write_part_manifest(video_id, [marker], keyframes=len(keyframes), merge=True)
        done_count += 1
        
    except Exception as err:
        failures.append((video_id, repr(err)))
        print(f"  [LỖI] {video_id}: {err}")
        traceback.print_exc(limit=3)
    finally:
        wav_path.unlink(missing_ok=True)
        map_csv.unlink(missing_ok=True)
        if not KEEP_MODELS:
            unload_models()

print("\n" + "=" * 72)
print(f"phiên này: {done_count} video, {len(failures)} lỗi, "
      f"{(time.perf_counter() - started) / 3600:.1f}h")
for video_id, err in failures:
    print(f"  {video_id}: {err}")


## Step 2 — Chốt phiên

In [ ]:
unload_models()

fresh = StageStatus.load(transport, layout, MEMBER)
left  = [(s, v) for v in MY_VIDEOS for s in STAGE_LIST if not fresh.is_done(s, v)]

total = len(MY_VIDEOS) * len(STAGE_LIST)
print(f"{MEMBER}: {total - len(left)}/{total} việc đã lên {VOLUME}")
for stage in STAGE_LIST:
    n = sum(1 for v in MY_VIDEOS if fresh.is_done(stage, v))
    print(f"  {stage:<16} {n:>4}/{len(MY_VIDEOS)} video")

if left:
    print("\nCÒN LẠI VIỆC — bấm Run All lại, nó sẽ tự lướt qua video đã làm và chạy tiếp từ chỗ đứt.")
else:
    print("\n🎉 HOÀN THÀNH XUẤT SẮC! PHẦN CỦA BẠN ĐÃ XONG! MỜI BÁO LEAD.")
